In [2]:
!pip install yfinance xgboost ta joblib

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=7f7061699cd9ef097e6838c13703d4a7ada494b3abc441e81e25041d80d7f484
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


In [3]:
import pandas as pd
import numpy as np
import yfinance as yf
import ta
import joblib
from datetime import datetime

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

In [4]:
stocks = [
    "ADANIENT.NS",
    "ADANIPORTS.NS",
    "APOLLOHOSP.NS",
    "ASIANPAINT.NS",
    "AXISBANK.NS",
    "BAJAJ-AUTO.NS",
    "BAJFINANCE.NS",
    "BAJAJFINSV.NS",
    "BEL.NS",
    "BHARTIARTL.NS",
    "CIPLA.NS",
    "COALINDIA.NS",
    "DRREDDY.NS",
    "EICHERMOT.NS",
    "ETERNAL.NS",
    "GRASIM.NS",
    "HCLTECH.NS",
    "HDFCBANK.NS",
    "HDFCLIFE.NS",
    "HEROMOTOCO.NS",
    "HINDALCO.NS",
    "HINDUNILVR.NS",
    "ICICIBANK.NS",
    "INDUSINDBK.NS",
    "INFY.NS",
    "ITC.NS",
    "JIOFIN.NS",
    "JSWSTEEL.NS",
    "KOTAKBANK.NS",
    "LT.NS",
    "M&M.NS",
    "MARUTI.NS",
    "NESTLEIND.NS",
    "NTPC.NS",
    "ONGC.NS",
    "POWERGRID.NS",
    "RELIANCE.NS",
    "SBILIFE.NS",
    "SBIN.NS",
    "SHRIRAMFIN.NS",
    "SUNPHARMA.NS",
    "TATACONSUM.NS",
    "TATAMOTORS.NS",
    "TATASTEEL.NS",
    "TCS.NS",
    "TECHM.NS",
    "TITAN.NS",
    "TRENT.NS",
    "ULTRACEMCO.NS",
    "WIPRO.NS"
]

In [5]:


# Get today's date dynamically (will resolve to '2026-06-17' today)
today_date = datetime.today().strftime("%Y-%m-%d")

all_data = []

for stock in stocks:
    try:
        print(f"Downloading {stock}...")

        # Updated end date to fetch up to today
        df = yf.download(
            stock,
            start="2014-01-01",
            end=today_date,
            progress=False,
            auto_adjust=True,
        )

        if len(df) == 0:
            print(f"No data found for {stock}")
            continue

        df = df.reset_index()

        # Clean multi-index columns properly
        df.columns = [
            col[0] if isinstance(col, tuple) and col[0] != "" else col
            for col in df.columns
        ]

        df["Stock"] = stock
        all_data.append(df)

    except Exception as e:
        print(f"Error downloading {stock}: {e}")

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['TATAMOTORS.NS']: YFTzMissingError('possibly delisted; no timezone found')


No data found for TATAMOTORS.NS


In [20]:
combined_df = pd.concat(
    all_data,
    ignore_index=True
)

print(combined_df.shape)
combined_df.tail()

(144464, 7)


,Date,Close,High,Low,Open,Volume,Stock
144459,2026-06-10,178.929993,182.289993,178.500000,182.100006,16935979,WIPRO.NS
144460,2026-06-11,177.369995,178.199997,175.830002,176.970001,14468882,WIPRO.NS
144461,2026-06-12,180.139999,180.449997,177.570007,178.649994,13850153,WIPRO.NS
144462,2026-06-15,181.380005,183.000000,180.910004,183.000000,13870660,WIPRO.NS
144463,2026-06-16,182.669998,183.380005,181.630005,182.389999,11706020,WIPRO.NS


In [7]:
def create_features(df):

    df = df.copy()

    close = df["Close"]
    volume = df["Volume"]

    for i in range(1, 11):

        df[f"Lag_{i}"] = close.shift(i)

    df["SMA_10"] = close.rolling(10).mean()
    df["SMA_20"] = close.rolling(20).mean()

    df["EMA_10"] = close.ewm(span=10).mean()
    df["EMA_20"] = close.ewm(span=20).mean()

    df["RSI"] = ta.momentum.RSIIndicator(close).rsi()

    macd = ta.trend.MACD(close)

    df["MACD"] = macd.macd()
    df["MACD_SIGNAL"] = macd.macd_signal()

    df["RETURN"] = close.pct_change()

    df["VOL_CHANGE"] = volume.pct_change()

    return df

In [8]:
feature_dfs = []

for stock in combined_df["Stock"].unique():

    temp = combined_df[
        combined_df["Stock"] == stock
    ].copy()

    temp = create_features(temp)

    feature_dfs.append(temp)

combined_df = pd.concat(
    feature_dfs,
    ignore_index=True
)

In [9]:
combined_df["Target"] = (
    combined_df.groupby("Stock")["Close"]
    .shift(-1)
)

In [10]:
le = LabelEncoder()

combined_df["Stock_ID"] = (
    le.fit_transform(
        combined_df["Stock"]
    )
)

In [11]:
combined_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

combined_df.dropna(
    inplace=True
)

combined_df.reset_index(
    drop=True,
    inplace=True
)

In [12]:
combined_df = combined_df.sort_values(
    by="Date"
)

In [13]:
features = [
    "Open",
    "High",
    "Low",
    "Volume",
    "SMA_10",
    "SMA_20",
    "EMA_10",
    "EMA_20",
    "RSI",
    "MACD",
    "MACD_SIGNAL",
    "RETURN",
    "VOL_CHANGE",
    "Stock_ID"
]

for i in range(1,11):

    features.append(
        f"Lag_{i}"
    )

In [14]:

split_index = int(
    len(combined_df) * 0.8
)

train_df = combined_df.iloc[
    :split_index
]


test_df = combined_df.iloc[
    split_index:
]

In [15]:


X_train = train_df[features]
y_train = train_df["Target"]

X_test = test_df[features]
y_test = test_df["Target"]

In [16]:
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [17]:
predictions = model.predict(
    X_test
)

In [18]:
mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

r2 = r2_score(
    y_test,
    predictions
)

mape = np.mean(
    np.abs(
        (y_test - predictions)
        / y_test
    )
) * 100

print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)
print("MAPE:", mape)

MAE : 139.82315799083085
RMSE: 619.1128779029158
R2  : 0.9497991078273657
MAPE: 2.2228273200215423


In [19]:
importance = pd.DataFrame({

    "Feature": features,

    "Importance":
    model.feature_importances_

})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

        Feature  Importance
2           Low    0.401027
1          High    0.258618
14        Lag_1    0.158127
6        EMA_10    0.031634
0          Open    0.031456
15        Lag_2    0.025899
17        Lag_4    0.025578
16        Lag_3    0.015761
19        Lag_6    0.011258
20        Lag_7    0.010722
18        Lag_5    0.009407
21        Lag_8    0.007047
4        SMA_10    0.005422
7        EMA_20    0.005084
23       Lag_10    0.001008
5        SMA_20    0.000836
22        Lag_9    0.000386
10  MACD_SIGNAL    0.000177
9          MACD    0.000125
8           RSI    0.000109
11       RETURN    0.000100
13     Stock_ID    0.000080
12   VOL_CHANGE    0.000074
3        Volume    0.000064


In [ ]:
joblib.dump(
    model,
    "stock_model.pkl"
)

joblib.dump(
    features,
    "feature_columns.pkl"
)

joblib.dump(
    le,
    "stock_encoder.pkl"
)

['stock_encoder.pkl']

In [ ]:
from google.colab import files

files.download(
    "stock_model.pkl"
)

files.download(
    "feature_columns.pkl"
)

files.download(
    "stock_encoder.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import yfinance as yf

stocks = [
    "RELIANCE.NS",
    "TCS.NS",
    "INFY.NS",
    "SBIN.NS"
]

for stock in stocks:

    df = yf.download(
        stock,
        start="2020-01-01",
        end="2026-01-01",
        auto_adjust=True
    )

    df.reset_index(inplace=True)

    filename = stock.replace(
        ".NS",
        ""
    ) + ".csv"

    df.to_csv(
        filename,
        index=False
    )

    print(f"Saved {filename}")

[*********************100%***********************]  1 of 1 completed


Saved RELIANCE.csv


[*********************100%***********************]  1 of 1 completed


Saved TCS.csv


[*********************100%***********************]  1 of 1 completed


Saved INFY.csv


[*********************100%***********************]  1 of 1 completed

Saved SBIN.csv


In [22]:
pip install optuna shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 14.6 MB/s eta 0:00:00


In [26]:
# ==========================================
# MINIMAL MODEL TRAINING CODE
# ==========================================

import warnings
import logging
import numpy as np
import pandas as pd
import joblib
from datetime import datetime
from pathlib import Path
import json

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import yfinance as yf
import ta
from xgboost import XGBRegressor
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ==========================================
# CONFIGURATION
# ==========================================

NIFTY_50_STOCKS = [
    "ADANIENT.NS", "ADANIPORTS.NS", "APOLLOHOSP.NS", "ASIANPAINT.NS",
    "AXISBANK.NS", "BAJAJ-AUTO.NS", "BAJFINANCE.NS", "BAJAJFINSV.NS",
    "BEL.NS", "BHARTIARTL.NS", "CIPLA.NS", "COALINDIA.NS",
    "DRREDDY.NS", "EICHERMOT.NS", "GRASIM.NS",
    "HCLTECH.NS", "HDFCBANK.NS", "HDFCLIFE.NS", "HEROMOTOCO.NS",
    "HINDALCO.NS", "HINDUNILVR.NS", "ICICIBANK.NS", "INDUSINDBK.NS",
    "INFY.NS", "ITC.NS", "JSWSTEEL.NS",
    "KOTAKBANK.NS", "LT.NS", "M&M.NS", "MARUTI.NS",
    "NESTLEIND.NS", "NTPC.NS", "ONGC.NS", "POWERGRID.NS",
    "RELIANCE.NS", "SBILIFE.NS", "SBIN.NS", "SHRIRAMFIN.NS",
    "SUNPHARMA.NS", "TATACONSUM.NS", "TATASTEEL.NS",
    "TCS.NS", "TECHM.NS", "TITAN.NS", "TRENT.NS",
    "ULTRACEMCO.NS", "WIPRO.NS"
]

START_DATE = "2014-01-01"
OUTPUT_DIR = Path("./stock_model_artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

# ==========================================
# DOWNLOAD DATA
# ==========================================

def download_stock_data(symbol, start_date, end_date):
    try:
        df = yf.download(symbol, start=start_date, end=end_date, progress=False, auto_adjust=True)
        if df is None or df.empty or len(df) < 100:
            return None
        df = df.reset_index()
        df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
        df['Stock'] = symbol
        return df
    except:
        return None

logger.info("Downloading stock data...")
today = datetime.today().strftime("%Y-%m-%d")
all_data = []

for stock in NIFTY_50_STOCKS:
    df = download_stock_data(stock, START_DATE, today)
    if df is not None:
        all_data.append(df)

data = pd.concat(all_data, ignore_index=True)
logger.info(f"Downloaded {len(all_data)} stocks, {len(data)} rows")

# ==========================================
# FEATURE ENGINEERING
# ==========================================

logger.info("Creating features...")

def create_features(df):
    df = df.copy()
    close = df['Close']
    high = df['High']
    low = df['Low']
    volume = df['Volume']

    # Moving averages
    for period in [5, 10, 20, 50]:
        df[f'SMA_{period}'] = close.rolling(window=period, min_periods=1).mean()
        df[f'EMA_{period}'] = close.ewm(span=period, adjust=False).mean()

    # Bollinger Bands
    bb = ta.volatility.BollingerBands(close, window=20, window_dev=2)
    df['BB_HIGH'] = bb.bollinger_hband()
    df['BB_LOW'] = bb.bollinger_lband()
    df['BB_WIDTH'] = (df['BB_HIGH'] - df['BB_LOW']) / (bb.bollinger_mavg() + 1e-10)

    # Momentum
    df['RSI'] = ta.momentum.RSIIndicator(close, window=14).rsi()
    macd = ta.trend.MACD(close)
    df['MACD'] = macd.macd()
    df['MACD_SIGNAL'] = macd.macd_signal()

    # Volatility
    df['ATR'] = ta.volatility.AverageTrueRange(high, low, close, window=14).average_true_range()
    df['ROLLING_VOL'] = close.pct_change().rolling(window=20).std()

    # Returns
    df['RETURN_1D'] = close.pct_change(1)
    df['RETURN_5D'] = close.pct_change(5)
    df['RETURN_10D'] = close.pct_change(10)

    # Lags
    for i in range(1, 6):
        df[f'LAG_CLOSE_{i}'] = close.shift(i)
        df[f'LAG_VOLUME_{i}'] = volume.shift(i)

    # Price patterns
    df['HIGH_LOW_RATIO'] = high / (low + 1e-10)
    df['PRICE_SMA_RATIO'] = close / (df['SMA_20'] + 1e-10)

    return df

feature_dfs = []
for stock in data['Stock'].unique():
    stock_df = data[data['Stock'] == stock].copy()
    stock_df = create_features(stock_df)
    feature_dfs.append(stock_df)

data = pd.concat(feature_dfs, ignore_index=True)

# ==========================================
# CREATE TARGET
# ==========================================

data = data.sort_values(['Stock', 'Date']).reset_index(drop=True)
data['Target'] = data.groupby('Stock')['Close'].shift(-1)

# ==========================================
# PREPROCESSING
# ==========================================

logger.info("Preprocessing...")

# Remove inf/nan
data.replace([np.inf, -np.inf], np.nan, inplace=True)
for stock in data['Stock'].unique():
    mask = data['Stock'] == stock
    data.loc[mask] = data.loc[mask].ffill().bfill()

# Encode stocks
label_encoder = LabelEncoder()
data['Stock_ID'] = label_encoder.fit_transform(data['Stock'])

# Select features
feature_cols = [col for col in data.columns
               if col not in ['Date', 'Stock', 'Target', 'Open', 'High', 'Low', 'Close', 'Volume']]

data = data[feature_cols + ['Target']].dropna()
logger.info(f"Features: {len(feature_cols)}, Samples: {len(data)}")

# ==========================================
# TRAIN/TEST SPLIT
# ==========================================

split_idx = int(len(data) * 0.8)
train_data = data.iloc[:split_idx]
test_data = data.iloc[split_idx:]

X_train = train_data[feature_cols].values
y_train = train_data['Target'].values
X_test = test_data[feature_cols].values
y_test = test_data['Target'].values

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

logger.info(f"Train: {len(X_train)}, Test: {len(X_test)}")

# ==========================================
# HYPERPARAMETER OPTIMIZATION
# ==========================================

logger.info("Optimizing hyperparameters...")

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
        'random_state': 42,
        'verbosity': 0
    }

    # Use validation split
    split = int(len(X_train) * 0.8)
    X_t, y_t = X_train[:split], y_train[:split]
    X_v, y_v = X_train[split:], y_train[split:]

    model = XGBRegressor(**params)
    model.fit(X_t, y_t, verbose=False)

    y_pred = model.predict(X_v)
    rmse = np.sqrt(mean_squared_error(y_v, y_pred))
    return rmse

study = optuna.create_study(
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5),
    direction='minimize'
)

study.optimize(objective, n_trials=30, show_progress_bar=True)

best_params = study.best_params
logger.info(f"Best RMSE: {study.best_value:.4f}")
logger.info(f"Best params: {best_params}")

# ==========================================
# TRAIN FINAL MODEL
# ==========================================

logger.info("Training final model...")

model = XGBRegressor(**best_params, random_state=42, verbosity=0)
model.fit(X_train, y_train, verbose=False)

# ==========================================
# EVALUATE
# ==========================================

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-5))) * 100

direction_true = np.diff(y_test) > 0
direction_pred = np.diff(y_pred) > 0
directional_acc = np.mean(direction_true == direction_pred) * 100

metrics = {
    'MAE': float(mae),
    'RMSE': float(rmse),
    'R2': float(r2),
    'MAPE': float(mape),
    'Directional_Accuracy': float(directional_acc)
}

logger.info("\n" + "="*50)
logger.info("MODEL METRICS")
logger.info("="*50)
for k, v in metrics.items():
    logger.info(f"{k}: {v:.4f}")
logger.info("="*50)

# ==========================================
# SAVE MODEL
# ==========================================

logger.info("Saving model...")

joblib.dump(model, OUTPUT_DIR / 'model.pkl')
joblib.dump(feature_cols, OUTPUT_DIR / 'features.pkl')
joblib.dump(scaler, OUTPUT_DIR / 'scaler.pkl')
joblib.dump(label_encoder, OUTPUT_DIR / 'encoder.pkl')

with open(OUTPUT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

# Feature importance
importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

importance.to_csv(OUTPUT_DIR / 'feature_importance.csv', index=False)
logger.info(f"\nTop 10 Features:\n{importance.head(10)}")

logger.info(f"\n✅ Model saved to {OUTPUT_DIR}/")
logger.info("✅ Training complete!")

[I 2026-06-17 14:10:41,622] A new study created in memory with name: no-name-83fdb2c7-d688-40af-9f95-432c1a9c15c4


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-06-17 14:10:54,536] Trial 0 finished with value: 19.495794419193608 and parameters: {'n_estimators': 362, 'learning_rate': 0.22648248189516848, 'max_depth': 8, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 2, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893}. Best is trial 0 with value: 19.495794419193608.
[I 2026-06-17 14:11:01,734] Trial 1 finished with value: 13.612009945800422 and parameters: {'n_estimators': 521, 'learning_rate': 0.05675206026988748, 'max_depth': 3, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'min_child_weight': 3, 'reg_alpha': 4.329370014459266e-07, 'reg_lambda': 4.4734294104626844e-07}. Best is trial 1 with value: 13.612009945800422.
[I 2026-06-17 14:11:09,360] Trial 2 finished with value: 11.68481255449004 and parameters: {'n_estimators': 313, 'learning_rate': 0.0199473547030745, 'max_depth': 6, 'subsample': 0.645614570099021, 'colsample_bytree': 0.805926447

In [27]:
import numpy as np
import pandas as pd
import joblib
import json
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configuration
OUTPUT_DIR = Path("./stock_model_artifacts")

def load_artifacts():
    """Loads all saved artifacts required for evaluation."""
    print("⏳ Loading model artifacts...")
    try:
        model = joblib.load(OUTPUT_DIR / 'model.pkl')
        feature_cols = joblib.load(OUTPUT_DIR / 'features.pkl')
        scaler = joblib.load(OUTPUT_DIR / 'scaler.pkl')
        label_encoder = joblib.load(OUTPUT_DIR / 'encoder.pkl')

        # Load the training-time metrics for comparison if available
        metrics_path = OUTPUT_DIR / 'metrics.json'
        trained_metrics = None
        if metrics_path.exists():
            with open(metrics_path, 'r') as f:
                trained_metrics = json.load(f)

        return model, feature_cols, scaler, label_encoder, trained_metrics
    except FileNotFoundError as e:
        print(f"❌ Error: Could not find all artifacts in {OUTPUT_DIR}. Did you run the training script first?")
        raise e

def print_saved_metrics(metrics):
    """Displays the metrics generated directly during training."""
    print("\n" + "="*50)
    print("📋 METRICS STORED DURING TRAINING")
    print("="*50)
    for k, v in metrics.items():
        if "Accuracy" in k or "MAPE" in k:
            print(f"{k:<25}: {v:.2f}%")
        else:
            print(f"{k:<25}: {v:.4f}")
    print("="*50)

if __name__ == "__main__":
    # 1. Load the artifacts
    model, feature_cols, scaler, label_encoder, trained_metrics = load_artifacts()

    if trained_metrics:
        print_saved_metrics(trained_metrics)
    else:
        print("⚠️ No metrics.json file found to print.")

    # 2. Display Feature Importance Summary
    importance_path = OUTPUT_DIR / 'feature_importance.csv'
    if importance_path.exists():
        print("\n🏆 TOP 5 MOST INFLUENTIAL FEATURES:")
        importance_df = pd.read_csv(importance_path)
        print(importance_df.head(5).to_string(index=False))
        print("="*50)

⏳ Loading model artifacts...

📋 METRICS STORED DURING TRAINING
MAE                      : 34.6645
RMSE                     : 163.7818
R2                       : 0.9941
MAPE                     : 2.29%
Directional_Accuracy     : 49.21%

🏆 TOP 5 MOST INFLUENTIAL FEATURES:
    Feature  Importance
      EMA_5    0.265860
     EMA_10    0.179220
LAG_CLOSE_2    0.128280
LAG_CLOSE_1    0.125696
     SMA_10    0.083965


In [29]:
import yfinance as yf

# Define the ticker symbol for TCS on the NSE
ticker = "TCS.NS"

# Set the start and end dates
# Note: yfinance end date is exclusive, so setting it to 2026-06-18
# ensures data for 2026-06-17 is fully included.
start_date = "2014-01-01"
end_date = "2026-06-18"

print(f"Downloading data for {ticker} from {start_date} to 2026-06-17...")

# Fetch the historical data
tcs_data = yf.download(ticker, start=start_date, end=end_date)

# Display the first few rows of the data
print("\nFirst 5 rows:")
print(tcs_data.head())

# Display the last few rows to verify it includes today's data
print("\nLast 5 rows:")
print(tcs_data.tail())

# Optional: Save the data to a CSV file
tcs_data.to_csv("TCS_stock_history.csv")
print("\nData successfully saved to 'TCS_stock_history.csv'")

[*********************100%***********************]  1 of 1 completed


First 5 rows:
Price            Close        High         Low        Open   Volume
Ticker          TCS.NS      TCS.NS      TCS.NS      TCS.NS   TCS.NS
Date                                                               
2014-01-01  800.301941  811.897808  799.484300  810.262527   529952
2014-01-02  805.393677  813.570262  801.361097  805.022014  1726948
2014-01-03  825.909302  828.473830  798.053185  804.538684  2618174
2014-01-06  832.376343  834.011623  816.543465  828.436679  2311810
2014-01-07  819.944153  838.638817  817.286799  832.524980  2897486

Last 5 rows:
Price             Close         High          Low    Open   Volume
Ticker           TCS.NS       TCS.NS       TCS.NS  TCS.NS   TCS.NS
Date                                                              
2026-06-11  2135.600098  2154.600098  2110.000000  2127.0  3079815
2026-06-12  2161.399902  2168.000000  2138.000000  2150.0  2124656
2026-06-15  2162.000000  2192.000000  2159.300049  2192.0  3337083
2026-06-16  2199.000000  